# 🔗 Link Resolver Pro
Automated multi-threaded link resolver using Selenium.  
Paste a URL, pick the buttons you want to click, and the script resolves the final download links in parallel.

In [ ]:
# =========================================
# 1. Install required packages
# =========================================
!pip install -q selenium

In [ ]:
# =========================================
# 2. Imports
# =========================================
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from multiprocessing import Pool
from urllib.parse import urlparse
import time

In [ ]:
# =========================================
# 3. Chrome options
# =========================================
def chrome_options():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    return options

In [ ]:
# =========================================
# 4. Scanner Helper (Finds all options)
# =========================================
def scan_page_for_buttons(url):
    print(f'🔍 Scanning page: {url} ...')
    driver = webdriver.Chrome(options=chrome_options())
    found_buttons = []

    try:
        driver.get(url)
        time.sleep(4)

        button_texts = ['Download Links', 'Episode Links', 'Batch/Zip File']
        raw_buttons = []
        for text in button_texts:
            elems = driver.find_elements(By.XPATH, f"//a[normalize-space()='{text}'] | //button[normalize-space()='{text}']")
            raw_buttons.extend(elems)

        unique_buttons = list(set(raw_buttons))
        unique_buttons.sort(key=lambda x: x.location['y'])

        print(f'✅ Found {len(unique_buttons)} buttons. Analyzing locations...')

        for index, btn in enumerate(unique_buttons):
            try:
                y_pos = btn.location['y']
                txt = btn.text
                found_buttons.append({
                    'id': index,
                    'text': txt,
                    'y': y_pos,
                    'url_hint': btn.get_attribute('href')
                })
            except:
                pass

    finally:
        driver.quit()

    return found_buttons

In [ ]:
# =========================================
# 5. Extract domain from pepe links
# =========================================
def extract_pepe_domain(driver):
    """Dynamically extract the domain that hosts pepe links from the current page"""
    try:
        links = driver.find_elements(By.TAG_NAME, 'a')
        pepe_domains = set()

        for link in links:
            href = link.get_attribute('href')
            if href and '?go=pepe-' in href:
                parsed = urlparse(href)
                domain = f'{parsed.scheme}://{parsed.netloc}'
                pepe_domains.add(domain)

        if pepe_domains:
            return list(pepe_domains)[0]
    except:
        pass

    return None

In [ ]:
# =========================================
# 6. Worker function (Executes Selection)
# =========================================
def worker(task):
    target_url, button_index_to_click = task
    print(f'🚀 [Worker] Starting task for Button #{button_index_to_click + 1}...')

    driver = webdriver.Chrome(options=chrome_options())
    wait = WebDriverWait(driver, 25)

    try:
        driver.get(target_url)
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
        time.sleep(3)

        button_texts = ['Download Links', 'Episode Links', 'Batch/Zip File']
        raw_buttons = []
        for text in button_texts:
            elems = driver.find_elements(By.XPATH, f"//a[normalize-space()='{text}'] | //button[normalize-space()='{text}']")
            raw_buttons.extend(elems)

        unique_buttons = list(set(raw_buttons))
        unique_buttons.sort(key=lambda x: x.location['y'])

        if button_index_to_click >= len(unique_buttons):
            return f'❌ [Worker] Button index {button_index_to_click} no longer exists.'

        target_btn = unique_buttons[button_index_to_click]

        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target_btn)
        time.sleep(1)

        current_handles = set(driver.window_handles)
        target_btn.click()
        print(f'✅ [Worker] Clicked button #{button_index_to_click + 1}')
        time.sleep(5)

        new_handles = set(driver.window_handles) - current_handles
        if new_handles:
            driver.switch_to.window(new_handles.pop())

        try:
            server_btn = wait.until(EC.presence_of_element_located((
                By.XPATH, "//*[contains(text(), 'Fast Server') or contains(text(), 'All Episodes Batch')]"
            )))
            driver.execute_script('arguments[0].click();', server_btn)
            time.sleep(3)
            if len(driver.window_handles) > len(current_handles) + 1:
                driver.switch_to.window(driver.window_handles[-1])
        except:
            pass

        def click_verify(txt):
            try:
                xpath = f"//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{txt.lower()}')]"
                el = wait.until(EC.presence_of_element_located((By.XPATH, xpath)))
                driver.execute_script('arguments[0].click();', el)
                print(f"[Worker] Clicked '{txt}'")
                return True
            except:
                return False

        click_verify('Start Verification')
        time.sleep(2)
        click_verify('Verify To Continue')
        print('[Worker] Waiting 5s...')
        time.sleep(5)
        click_verify('Click Here To Continue')
        time.sleep(2)
        click_verify('Go to download')

        print('[Worker] Searching for final pepe link...')
        time.sleep(5)

        pepe_domain = extract_pepe_domain(driver)

        if not pepe_domain:
            return '❌ [Worker] Failed: Could not find pepe domain on page'

        print(f'[Worker] Detected pepe domain: {pepe_domain}')
        target_pattern = f'{pepe_domain}/?go=pepe-'

        found_link = None
        links = driver.find_elements(By.TAG_NAME, 'a')
        for link_element in links:
            href = link_element.get_attribute('href')
            if href and href.startswith(target_pattern):
                found_link = href
                break

        if found_link:
            driver.get(found_link)
            time.sleep(3)
            final_resolved_url = driver.current_url
            print(f'🎯 [Worker] FINAL RESULT: {final_resolved_url}')
            return final_resolved_url
        else:
            return '❌ [Worker] Failed: No matching pepe link found'

    except Exception as e:
        return f'❌ Error [Worker]: {str(e)}'

    finally:
        driver.quit()

In [ ]:
# =========================================
# 7. Main execution (INTERACTIVE)
# =========================================
start_url = input('Paste your link here and press Enter: ').strip()

if not start_url:
    print('No URL provided')
else:
    buttons = scan_page_for_buttons(start_url)

    if not buttons:
        print('❌ No buttons found on this page.')
    else:
        print('\n' + '='*40)
        print('👇 AVAILABLE BUTTONS FOUND 👇')
        print('='*40)

        for btn in buttons:
            print(f"[{btn['id'] + 1}] {btn['text']}  (Y-Pos: {btn['y']})")

        print('='*40)
        print('Which buttons do you want to click?')
        print("Type numbers separated by commas (e.g., '1, 3' or just '2')")

        selection = input('Enter selection: ').strip()

        selected_indices = []
        try:
            parts = selection.split(',')
            for p in parts:
                num = int(p.strip())
                idx = num - 1
                if 0 <= idx < len(buttons):
                    selected_indices.append(idx)
                else:
                    print(f'⚠️ Skipping invalid number: {num}')
        except:
            print('❌ Invalid input format.')

        if not selected_indices:
            print('No valid buttons selected. Exiting.')
        else:
            tasks = [(start_url, idx) for idx in selected_indices]

            print(f'\n🚀 Launching {len(tasks)} parallel workers...')

            try:
                with Pool(processes=len(tasks)) as pool:
                    results = pool.map(worker, tasks)

                print('\n' + '='*60)
                print('--- FINAL RESULTS ---')
                print('='*60)
                for i, r in enumerate(results, 1):
                    print(f'[{i}] {r}')
                print('='*60)
            except Exception as e:
                print(f'❌ Execution Error: {e}')